# Task 2: Define & Register Tools
In this task, three LangChain tools will be created and registered using the `@tool` decorator. Two tools will be reused from the Day 5 raw Python agent: a calculator tool and a weather tool. A new product-price tool will also be created to read real data from a local CSV file.
The tools will have clear docstrings so the language model can understand their purpose and decide when to use them.

In [19]:
import pandas as pd
from langchain_core.tools import tool

products = pd.DataFrame({
    "product": ["Laptop A", "Laptop B", "Smartphone A", "Headphones A"],
    "price": [85000, 120000, 65000, 15000],
    "currency": ["PKR", "PKR", "PKR", "PKR"]
})

products.to_csv("products.csv", index=False)

print("products.csv created successfully.")
print(products)

# Calculator Tool
@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Perform addition, subtraction, multiplication, or division on two numbers."""

    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b
    else:
        raise ValueError("Unknown operation.")

# Weather Tool
@tool
def get_weather(city: str) -> dict:
    """Return the current weather information for a supported city."""

    weather_data = {
        "karachi": {"temperature": 34, "condition": "Sunny"},
        "lahore": {"temperature": 31, "condition": "Partly cloudy"},
        "islamabad": {"temperature": 27, "condition": "Cloudy"},
        "dubai": {"temperature": 38, "condition": "Sunny"}
    }

    city_key = city.strip().lower()

    if city_key not in weather_data:
        raise ValueError(f"Weather data not available for {city}.")

    return weather_data[city_key]

# Product Price Tool
@tool
def get_product_price(product_name: str) -> dict:
    """Return the price and currency of a product from the local CSV file."""
    df = pd.read_csv("products.csv")
    match = df[df["product"].str.lower() == product_name.strip().lower()]

    if match.empty:
        raise ValueError(f"Product '{product_name}' not found in the catalog.")

    row = match.iloc[0]
    return {
        "product": row["product"],
        "price": int(row["price"]),
        "currency": row["currency"]
    }


products.csv created successfully.
        product   price currency
0      Laptop A   85000      PKR
1      Laptop B  120000      PKR
2  Smartphone A   65000      PKR
3  Headphones A   15000      PKR


In [20]:
tools = [
    calculator,
    get_weather,
    get_product_price
]

print("Registered tools:")

for tool_item in tools:
    print("-", tool_item.name)
    

Registered tools:
- calculator
- get_weather
- get_product_price


In [21]:
print("Calculator:")
print(calculator.invoke({
    "a": 20,
    "b": 5,
    "operation": "multiply"
}))

print("\nWeather:")
print(get_weather.invoke({
    "city": "Karachi"
}))

print("\nProduct Price:")
print(get_product_price.invoke({
    "product_name": "Laptop A"
}))

Calculator:
100.0

Weather:
{'temperature': 34, 'condition': 'Sunny'}

Product Price:
{'product': 'Laptop A', 'price': 85000, 'currency': 'PKR'}


## Tool Docstrings

The docstring of each LangChain tool describes what the tool does. LangChain uses this description as tool metadata that is provided to the language model, helping the model decide when and how to use the tool.
Therefore, tool docstrings are important because they become part of the tool context available to the model. Clear and specific descriptions improve tool selection and reduce incorrect tool usage.

In [22]:
for tool_item in tools:
    print("Tool:", tool_item.name)
    print("Description:", tool_item.description)
    print("Schema:", tool_item.args_schema)
    print("-" * 60)

Tool: calculator
Description: Perform addition, subtraction, multiplication, or division on two numbers.
Schema: <class 'langchain_core.utils.pydantic.calculator'>
------------------------------------------------------------
Tool: get_weather
Description: Return the current weather information for a supported city.
Schema: <class 'langchain_core.utils.pydantic.get_weather'>
------------------------------------------------------------
Tool: get_product_price
Description: Return the price and currency of a product from the local CSV file.
Schema: <class 'langchain_core.utils.pydantic.get_product_price'>
------------------------------------------------------------


In [23]:
calculator_result = calculator.invoke({
    "a": 25,
    "b": 5,
    "operation": "multiply"
})

print("Calculator Result:", calculator_result)

Calculator Result: 125.0


In [24]:
weather_result = get_weather.invoke({
    "city": "Karachi"
})

print("Weather Result:", weather_result)

Weather Result: {'temperature': 34, 'condition': 'Sunny'}


In [25]:
# Test Csv
price_result = get_product_price.invoke({
    "product_name": "Laptop A"
})

print("Product Price Result:", price_result)

Product Price Result: {'product': 'Laptop A', 'price': 85000, 'currency': 'PKR'}


In [26]:
# Test Error Handling
try:
    get_product_price.invoke({
        "product_name": "Unknown Product"
    })
except Exception as e:
    print("Tool Error Handled:", e)

Tool Error Handled: Product 'Unknown Product' not found in the catalog.


In [27]:
# Test  Devision by Zero
try:
    calculator.invoke({
        "a": 10,
        "b": 0,
        "operation": "divide"
    })
except Exception as e:
    print("Calculator Error Handled:", e)

Calculator Error Handled: Cannot divide by zero.


## Why Tool Docstrings Matter

LangChain uses the tool name and docstring as metadata that is provided to the language model. The description tells the model what the tool does and helps it select the correct tool for a user's request.
Clear docstrings are therefore important because they act as instructions for tool selection. Poor or ambiguous descriptions can cause the model to choose the wrong tool or provide incorrect arguments.

## Raw Python vs LangChain Tools

In the Day 5 raw Python agent, tools were stored as normal Python functions inside a manual tool registry. In LangChain, the tool decorator converts functions into structured tools with names, descriptions, and input schemas.
This reduces the amount of manual tool-registration code and makes the tools easier to connect with an agent.
## Task 2 Conclusion

Three tools were successfully created and registered using LangChain's `@tool` decorator. The calculator and weather tools were reused from the Day 5 raw Python agent, while the new product-price tool reads real data from a local CSV file.

All tools were tested with successful inputs and failure cases. Tool names, descriptions, and input schemas were also inspected to verify that LangChain converted the Python functions into structured tools suitable for agent use.